# 24 · 评估 4 件套 + RAG 类封装

> **学习目标**：手撸 RAGAS 风格的 4 个核心指标（faithfulness / context_relevance / answer_relevance / context_precision），跑 30 题 eval set，把 RAG 包成一个可被外部代码调用的 class。
>
> **预备**：16–23 跑过。**ragas / fastapi 都装不上 ── 全手撸**。
>
> **为什么重要**：没有评估的 RAG = 黑盒赌博。**这是 RAG 工程师水平的真正分水岭**：能写一份 baseline 表 → 才有迭代节奏。

In [ ]:
MODE = 'OFFLINE'

import numpy as np, hashlib, re, json, time, requests
from dataclasses import dataclass, field
from typing import Callable
OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text, dim=256):
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    v = np.random.default_rng(seed).standard_normal(dim).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_embed(text):
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model':'nomic-embed-text','prompt':text}, timeout=30)
    r.raise_for_status()
    v = np.array(r.json()['embedding'], dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_chat(prompt, temp=0.0):
    r = requests.post(f'{OLLAMA}/api/chat', json={
        'model':'qwen1.5_1.8','stream':False,'options':{'temperature':temp},
        'messages':[{'role':'user','content':prompt}]
    }, timeout=120)
    r.raise_for_status()
    return r.json()['message']['content']

if MODE == 'ONLINE':
    try: requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status(); embed = ollama_embed; chat = ollama_chat; print('✅ ONLINE')
    except Exception: MODE='OFFLINE'
if MODE == 'OFFLINE':
    embed = fake_embed
    def chat(p): return f'[STUB-LLM] {p[:60]}'
    print('OFFLINE：metric 用规则评分；ONLINE 切到 LLM-as-judge')

## 1. RAG 类封装 —— 把所有组件挂在一个 callable 上

**设计原则**：
- 组件**可注入**：embed_fn、chat_fn、reranker_fn 都从外部传
- 一个 `.query(question)` 方法把整条 pipeline 跑通
- 返回结构化结果：`answer` + `contexts` + `trace`（方便评估）

In [ ]:
@dataclass
class RAGResult:
    question: str
    answer: str
    contexts: list[str]
    retrieved_ids: list[int]
    latency_ms: float

class RAGSystem:
    PROMPT = '''你是助手。仅基于上下文回答；context 不够则说"我不知道"。
上下文:
{context}

问题: {question}
回答:'''

    def __init__(self, docs: list[str], embed_fn: Callable, chat_fn: Callable,
                 reranker_fn: Callable | None = None, top_k: int = 3):
        self.docs = docs
        self.embed_fn = embed_fn
        self.chat_fn = chat_fn
        self.reranker_fn = reranker_fn
        self.top_k = top_k
        self.doc_vecs = np.vstack([embed_fn(d) for d in docs])

    def _retrieve(self, question: str, recall_k: int) -> list[int]:
        q = self.embed_fn(question)
        sims = self.doc_vecs @ q
        return [int(i) for i in np.argsort(-sims)[:recall_k]]

    def query(self, question: str) -> RAGResult:
        t0 = time.perf_counter()
        # 召回放大
        recall_k = self.top_k * (3 if self.reranker_fn else 1)
        candidate_ids = self._retrieve(question, recall_k)
        # rerank（可选）
        if self.reranker_fn:
            scored = self.reranker_fn(question, [self.docs[i] for i in candidate_ids])
            # scored 是 [(idx_in_candidates, score)] —— 按分数取 top_k
            top_local = sorted(scored, key=lambda x: -x[1])[:self.top_k]
            final_ids = [candidate_ids[i] for i, _ in top_local]
        else:
            final_ids = candidate_ids[:self.top_k]
        contexts = [self.docs[i] for i in final_ids]
        # generate
        prompt = self.PROMPT.format(context='\n'.join(f'[d{i}] {c}' for i, c in zip(final_ids, contexts)), question=question)
        answer = self.chat_fn(prompt)
        return RAGResult(question=question, answer=answer, contexts=contexts,
                          retrieved_ids=final_ids, latency_ms=(time.perf_counter()-t0)*1000)

# 简易 stub reranker：token 重叠
def stub_reranker(query, docs):
    q_tokens = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', query.lower()))
    out = []
    for i, d in enumerate(docs):
        d_tokens = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', d.lower()))
        out.append((i, float(len(q_tokens & d_tokens))))
    return out

## 2. 30 题 eval set —— 多样性是关键

**eval set 设计原则**：
- 覆盖**易 / 中 / 难** 3 档
- 覆盖**精确关键词查询 / 同义改写 / 多跳推理 / 拒答场景** 4 类
- 每题标 `expected_doc_ids`（用于检索段评估） + `gold_answer` 关键词（用于答案段评估）

In [ ]:
# 复用前面 notebook 同款的小知识库
DOCS = [
    '锂电池 XZ4054H 是一款高能量密度电池，容量 300mAh，循环寿命 500 次。',
    'XZ5352R 是一款 800mAh 锂电池，专为高放电场景设计。',
    'XZ4054H-NE1.11 是 XZ4054H 的升级版，集成保护电路，规格书 V1.11。',
    '电源管理芯片 PM8916 提供多路稳压输出，工作电压 3.3V。',
    'Transformer 是 2017 年由 Google 提出的注意力机制神经网络。',
    'RAG 是检索增强生成的简称，把外部知识接进 LLM 上下文。',
    '注意力机制（Attention）解决了 RNN 在长序列上的梯度衰减问题。',
    'PM8917 是 PM8916 的低功耗替代型号，待机电流降低 50%。',
    'LoRA 是参数高效微调方法，仅训练低秩矩阵 A 和 B。',
    'QLoRA 在 LoRA 基础上把 base 模型 4-bit 量化，单卡可调 7B。',
    'DPO 是无需 reward model 的偏好优化算法。',
    'Chroma 是本地友好的开源向量库，默认 HNSW 索引。',
    'Qdrant 用 Rust 实现，性能优秀，支持稀疏-稠密混合检索。',
    'HNSW 把检索复杂度从 O(N) 降到 O(log N)。',
    'BPE 是字节对编码，被 GPT-2/3/4 等所有现代 LLM 用作分词器。',
]

EVAL = [
    # 易：精确关键词
    {'q':'XZ4054H 容量是多少','expect':[0],'gold':['300','mAh']},
    {'q':'XZ5352R 多少 mAh','expect':[1],'gold':['800']},
    {'q':'PM8916 工作电压','expect':[3],'gold':['3.3','V']},
    {'q':'PM8917 是什么','expect':[7],'gold':['低功耗','PM8916']},
    {'q':'LoRA 训练什么','expect':[8],'gold':['低秩','A','B']},
    {'q':'DPO 算法','expect':[10],'gold':['偏好','reward']},
    {'q':'Chroma 向量库','expect':[11],'gold':['HNSW']},
    {'q':'BPE 分词器','expect':[14],'gold':['字节对','GPT']},
    # 中：同义改写
    {'q':'锂电池规格','expect':[0,1,2],'gold':['容量','mAh']},
    {'q':'微调的方法','expect':[8,9],'gold':['LoRA']},
    {'q':'注意力网络','expect':[4,6],'gold':['Transformer','注意力']},
    {'q':'检索增强','expect':[5],'gold':['RAG','外部知识']},
    {'q':'近似最近邻','expect':[13],'gold':['HNSW','log']},
    {'q':'量化微调','expect':[9],'gold':['QLoRA','4-bit']},
    # 中：上下游关系
    {'q':'XZ4054H 的升级版是什么','expect':[2],'gold':['XZ4054H-NE1.11','保护电路']},
    {'q':'PM 系列省电的型号','expect':[3,7],'gold':['PM8917','低功耗']},
    {'q':'QLoRA 和 LoRA 关系','expect':[8,9],'gold':['量化','低秩']},
    # 难：多跳 + 综合
    {'q':'什么是参数高效微调','expect':[8,9],'gold':['LoRA','低秩']},
    {'q':'神经网络注意力机制起源','expect':[4],'gold':['Transformer','Google','2017']},
    {'q':'向量数据库性能优化','expect':[11,12,13],'gold':['HNSW','Qdrant']},
    {'q':'高放电场景电池选型','expect':[1],'gold':['XZ5352R','800']},
    {'q':'Rust 写的向量库','expect':[12],'gold':['Qdrant']},
    # 拒答（doc 里没有）
    {'q':'今天纽约的天气','expect':[],'gold':['不知道','无法']},
    {'q':'2024 年的奥运会冠军','expect':[],'gold':['不知道','无法']},
    {'q':'XZ9999X 的规格','expect':[],'gold':['不知道','无法']},
    # 易：补充
    {'q':'Transformer 哪年提出','expect':[4],'gold':['2017']},
    {'q':'循环寿命 500','expect':[0],'gold':['XZ4054H']},
    {'q':'HNSW 复杂度','expect':[13],'gold':['log']},
    {'q':'Qdrant 编程语言','expect':[12],'gold':['Rust']},
    {'q':'GPT 用什么分词器','expect':[14],'gold':['BPE','字节对']},
]
print(f'eval set: {len(EVAL)} 题')
print(f'  精确关键词:  {sum(1 for e in EVAL if len(e["expect"]) == 1 and len(e["q"]) < 15)}')
print(f'  同义/多跳:    {sum(1 for e in EVAL if len(e["expect"]) >= 2)}')
print(f'  拒答:          {sum(1 for e in EVAL if not e["expect"])}')

## 3. 4 个核心指标 —— 全部手撸

**OFFLINE 模式用规则；ONLINE 模式用 LLM-as-judge**。两种模式的接口完全一样。

### 3.1 检索段指标

In [ ]:
# 检索段指标 1：context_relevance —— 召回的 context 和 query 多相关
def context_relevance(question: str, contexts: list[str]) -> float:
    """0–1：召回的每个 context 平均相关度"""
    if not contexts: return 0.0
    if MODE == 'ONLINE':
        scores = []
        for ctx in contexts:
            resp = chat(f'文档与问题的相关度，0-10 整数评分，只输出数字：\n问题：{question}\n文档：{ctx}')
            m = re.search(r'\d+', resp)
            scores.append((float(m.group()) if m else 5.0) / 10.0)
        return float(np.mean(scores))
    # OFFLINE：token 重叠率
    q_t = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', question.lower()))
    if not q_t: return 0.0
    rels = []
    for ctx in contexts:
        c_t = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', ctx.lower()))
        rels.append(len(q_t & c_t) / len(q_t))
    return float(np.mean(rels))

# 检索段指标 2：context_precision —— top-k 里 "真相关" 的比例（基于 expected_ids 算）
def context_precision(retrieved_ids: list[int], expected_ids: list[int]) -> float:
    if not retrieved_ids:
        return 0.0
    if not expected_ids:
        # 拒答场景：召回应该「不命中所有 expected」，但 expected 是空，无意义
        return float('nan')
    expected_set = set(expected_ids)
    hits = sum(1 for r in retrieved_ids if r in expected_set)
    return hits / len(retrieved_ids)

In [ ]:
# 生成段指标 1：faithfulness —— 答案是否「忠于上下文」（不胡编）
def faithfulness(answer: str, contexts: list[str]) -> float:
    """0–1：答案中的「事实断言」有多少能在 context 里找到"""
    if MODE == 'ONLINE':
        ctx_text = '\n'.join(contexts)
        prompt = f'''请判断「回答」中的事实是否完全可由「上下文」推出。
评分 0-10：10 = 完全忠实；5 = 有部分编造；0 = 完全编造或无关。只输出数字。
上下文：{ctx_text}
回答：{answer}
分数：'''
        m = re.search(r'\d+', chat(prompt))
        return (float(m.group()) if m else 5.0) / 10.0
    # OFFLINE：答案 token 在 context 里出现的比例（粗略）
    ctx_t = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', ' '.join(contexts).lower()))
    a_t = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', answer.lower()))
    if not a_t:
        return 0.0
    # 移除常见停用词（避免「的、是、有」干扰）
    stop = set('的是有在了和与及对将让而为')
    a_t = a_t - stop
    if not a_t:
        return 1.0
    return len(a_t & ctx_t) / len(a_t)

# 生成段指标 2：answer_relevance —— 答案是否真的回答了 query
def answer_relevance(question: str, answer: str, gold_keywords: list[str]) -> float:
    """0–1：答案中含 gold 关键词的比例。若是拒答题，gold 含「不知道」/「无法」时判为对。"""
    if not gold_keywords: return float('nan')
    answer_lower = answer.lower()
    hits = sum(1 for k in gold_keywords if k.lower() in answer_lower)
    return hits / len(gold_keywords)

## 4. 把 4 指标串成评估循环

In [ ]:
def evaluate_rag(system: RAGSystem, eval_set: list[dict]) -> dict:
    rows = []
    for item in eval_set:
        res = system.query(item['q'])
        metrics = {
            'context_relevance': context_relevance(item['q'], res.contexts),
            'context_precision': context_precision(res.retrieved_ids, item['expect']),
            'faithfulness':       faithfulness(res.answer, res.contexts),
            'answer_relevance':  answer_relevance(item['q'], res.answer, item['gold']),
            'latency_ms':         res.latency_ms,
        }
        rows.append({**item, **metrics, 'retrieved': res.retrieved_ids, 'answer': res.answer[:80]})
    # 聚合（nan 排除）
    agg = {}
    for k in ['context_relevance', 'context_precision', 'faithfulness', 'answer_relevance', 'latency_ms']:
        vals = [r[k] for r in rows if isinstance(r[k], (int, float)) and not np.isnan(r[k])]
        agg[k] = float(np.mean(vals)) if vals else float('nan')
    return {'agg': agg, 'rows': rows}

# Baseline：朴素 RAG（无 rerank）
baseline = RAGSystem(DOCS, embed, chat, reranker_fn=None, top_k=3)
baseline_eval = evaluate_rag(baseline, EVAL)

print('===== Baseline (no rerank) =====')
for k, v in baseline_eval['agg'].items():
    suffix = ' ms' if 'latency' in k else ''
    print(f'  {k:<20} = {v:.3f}{suffix}')

In [ ]:
# 变体：加 reranker
variant = RAGSystem(DOCS, embed, chat, reranker_fn=stub_reranker, top_k=3)
variant_eval = evaluate_rag(variant, EVAL)

print('===== Variant (+ stub reranker) =====')
for k, v in variant_eval['agg'].items():
    suffix = ' ms' if 'latency' in k else ''
    print(f'  {k:<20} = {v:.3f}{suffix}')

print('\n===== Δ (variant - baseline) =====')
for k in baseline_eval['agg']:
    delta = variant_eval['agg'][k] - baseline_eval['agg'][k]
    sign = '+' if delta > 0 else ''
    bad_if_pos = 'latency' in k
    arrow = '🔺' if (delta > 0) ^ bad_if_pos else '🔻'
    print(f'  {k:<20} = {sign}{delta:+.3f}  {arrow}')
print('\n→ 这就是「baseline.md」格式：每次改动写一列，跟踪指标变化。')

In [ ]:
# 哪些 case baseline 错、variant 对（或反之）—— 失败案例分析
print('failed by baseline 但 variant 对的 case：')
for b, v in zip(baseline_eval['rows'], variant_eval['rows']):
    b_hit = (b['context_precision'] if not np.isnan(b['context_precision']) else 0) >= 0.5
    v_hit = (v['context_precision'] if not np.isnan(v['context_precision']) else 0) >= 0.5
    if not b_hit and v_hit:
        print(f'  Q: {b["q"]:<25} expect={b["expect"]} baseline={b["retrieved"]} variant={v["retrieved"]}')

print('\nbaseline 对但 variant 错的 case：')
for b, v in zip(baseline_eval['rows'], variant_eval['rows']):
    b_hit = (b['context_precision'] if not np.isnan(b['context_precision']) else 0) >= 0.5
    v_hit = (v['context_precision'] if not np.isnan(v['context_precision']) else 0) >= 0.5
    if b_hit and not v_hit:
        print(f'  Q: {b["q"]:<25} expect={b["expect"]} baseline={b["retrieved"]} variant={v["retrieved"]}')

## 5. baseline.md 模板 —— 你应该长期维护的一份文件

**生产 RAG 的 baseline.md 一般长这样**：

```markdown
# RAG eval baseline

eval_set: eval/v1_30q.jsonl   (30 题)

| date       | variant                       | ctx_rel | ctx_prec | faith | ans_rel | latency | notes |
|------------|-------------------------------|---------|----------|-------|---------|---------|-------|
| 2026-01-15 | baseline (dense top-3)        | 0.41    | 0.62     | 0.83  | 0.45    | 120 ms  | initial |
| 2026-01-18 | + chunk_size 500→300          | 0.45    | 0.71     | 0.85  | 0.48    | 125 ms  | +recall |
| 2026-01-21 | + hybrid BM25+RRF             | 0.48    | 0.78     | 0.86  | 0.52    | 145 ms  | hybrid wins on 关键词类 |
| 2026-01-25 | + LLM rerank (top 10→3)       | 0.55    | 0.85     | 0.88  | 0.61    | 2.1 s   | 延迟激增，但召回 +5% |
| 2026-02-01 | + bge-reranker（替 LLM rerank）| 0.54    | 0.84     | 0.87  | 0.60    | 165 ms  | 同样效果延迟 / 12x |
```

**真正的 RAG 工程师**：每个改动都写一行，**有数据支持地」迭代**。

**不是这样的**：「我感觉加了 rerank 之后好像更好了」← 这种是赌博，不是工程。

In [ ]:
# 生成一份 baseline.md 样例（实战中会按时间累积写）
from datetime import date

rows = [
    {'variant': 'baseline (dense top-3)',          'eval': baseline_eval['agg']},
    {'variant': '+ stub reranker (recall*3)',       'eval': variant_eval['agg']},
]
today = date.today().isoformat()

header = '| date | variant | ctx_rel | ctx_prec | faith | ans_rel | latency |'
sep    = '|------|---------|---------|----------|-------|---------|---------|'
lines  = [header, sep]
for r in rows:
    e = r['eval']
    lines.append(
        f'| {today} | {r["variant"]:<32} | {e["context_relevance"]:.3f} | {e["context_precision"]:.3f} | '
        f'{e["faithfulness"]:.3f} | {e["answer_relevance"]:.3f} | {e["latency_ms"]:.0f} ms |'
    )
print('\n'.join(lines))

# 实际生产里：把上面这个 string 追加到 `eval/baseline.md`
print('\n→ 把以上 markdown 表 append 到 eval/baseline.md，**每次改 RAG 配置都跑一遍 + 加一行**。')

## 6. 把 RAGSystem 包装成可被外部调用（HTTP 服务的 1 行准备）

**本机 fastapi 装不上**。但我们的 `RAGSystem.query(question)` 已经是一个 **纯函数式 API**：输入字符串，输出 RAGResult。要变成 HTTP 服务，差的只是「最薄的传输层」。

**伪代码**（实际部署在 [06-工程化与部署](../../../06-工程化与部署-Deployment/) 章节学）：

```python
# fastapi 版（装上后 6 行）
from fastapi import FastAPI
app = FastAPI()
rag = RAGSystem(docs=load_docs(), embed_fn=ollama_embed, chat_fn=ollama_chat)

@app.post('/query')
def query(payload: dict):
    return rag.query(payload['question']).__dict__
```

**stdlib http.server 版**（不装任何东西就能跑）：

In [ ]:
# stdlib HTTP 服务：5 行核心代码
from http.server import BaseHTTPRequestHandler
import json as _json

_GLOBAL_RAG = baseline   # 任意一个 RAGSystem 实例

class RAGHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        n = int(self.headers.get('Content-Length', 0))
        payload = _json.loads(self.rfile.read(n).decode('utf-8'))
        result = _GLOBAL_RAG.query(payload['question'])
        body = _json.dumps({
            'answer': result.answer, 'contexts': result.contexts,
            'retrieved_ids': result.retrieved_ids, 'latency_ms': result.latency_ms,
        }, ensure_ascii=False).encode('utf-8')
        self.send_response(200); self.send_header('Content-Type','application/json; charset=utf-8')
        self.end_headers(); self.wfile.write(body)
    def log_message(self, *a): pass  # 静音

print('上面 10 行就是一个最简 HTTP 服务的全部 handler。生产里：')
print('  - 加路由、Schema 校验 → 用 FastAPI / Starlette')
print('  - 异步 / 流式 / 多副本 / 监控 → 见 06-Deployment 章节')
print('  - 但**核心业务逻辑**（RAGSystem）已经写完，HTTP 只是「外壳」。')

## 深入思考

1. **OFFLINE 模式的「token 重叠」是真评估吗？**
   - 不是真评估，但**够做回归测试**：「这次改动相比上次，token 重叠率是涨是跌」就有信号。生产里换 LLM-as-judge / bge-reranker 评分。
2. **4 个指标里哪个最重要？**
   - 看业务。**事实型问答**（客服、知识库）→ faithfulness 最关键（不能编）。**总结型任务** → answer_relevance。**RAG debug** → context_precision 优先（看是不是召回烂）。
3. **30 题够吗？**
   - 起步够。**生产建议 100-200 题**，覆盖 long-tail 长尾问题。**关键不是题多，而是题覆盖度好**。
4. **`context_precision` 算 nan 的拒答题怎么算总分？**
   - 单独算「拒答正确率」：模型说「不知道」就对，否则错。**不要混进 hit@k 平均**。
5. **LLM-as-judge 评分会不会有偏？**
   - 会。已知偏：偏好长答案、偏好自己生成的风格、偏好礼貌话术。缓解：(a) rubric 严格；(b) pairwise 比较（A vs B 哪个好）；(c) 用多 LLM 投票。

**改一改**：
- 在 ONLINE 模式跑全套，看 OFFLINE rule-based 与 LLM-as-judge 的指标相关性（应该正相关但不严格）
- 把 `top_k` 改成 1 / 5，看 context_precision 如何变化

## 自检 ✅

- [ ] 默写 4 个核心指标各自衡量什么（context_relevance / context_precision / faithfulness / answer_relevance）。
- [ ] 解释「为什么必须有 baseline.md」。
- [ ] 不查代码画出 RAGSystem 的 4 个组件（embed / chat / rerank / top_k）。
- [ ] 给你一个失败 case，能根据 4 指标定位「是检索段问题还是生成段问题」。
- [ ] 解释「为什么 HTTP 不是 RAG 的本质 —— 业务逻辑应封装成 class」。

## 🎉 01-RAG 全部完成

**走完 16-24 你应该具备**：
- ✅ 50 行内手撸朴素 RAG
- ✅ 知道 chunking / embedding / hybrid / rerank / HyDE / Self-RAG 各自解决什么、不该解决什么
- ✅ 能为任意 RAG 项目写出 30 题 eval set + 4 指标 baseline.md
- ✅ 能看懂 rag_project / langchain / llama-index 任意一份代码

**下一步**：→ 进入 [02-Agent](../../../02-Agent/) 或 [04-模型微调](../../../04-模型微调-Finetuning/)